# Pré-processamento dos Dados de Consumo Elétrico — Aveiro
Este notebook realiza a limpeza, validação, deteção de outliers, normalização e engenharia de features
sobre os dados horários de consumo elétrico brutos, exportando um CSV pronto para Streamlit / Power BI.

In [5]:
# =============================================================
# CÉLULA 1 — IMPORTAÇÃO DE BIBLIOTECAS
# =============================================================
import pandas as pd
import numpy as np
from scipy import stats

print("Bibliotecas importadas com sucesso.")


Bibliotecas importadas com sucesso.


## 1. Carregamento e Unificação dos Dados Brutos

In [6]:
# Carregar os dois ficheiros com separador ponto e vírgula
df_3800 = pd.read_csv("consumoshorario_cp7_3800.csv", sep=";")
df_3810 = pd.read_csv("consumoshorario_cp7_3810.csv", sep=";")

# Unificar num único DataFrame
df_raw = pd.concat([df_3800, df_3810], ignore_index=True)

# Normalizar nomes de colunas (lowercase, sem acentos nem caracteres especiais)
df_raw.columns = (
    df_raw.columns
    .str.lower()
    .str.strip()
    .str.replace("í", "i", regex=False)
    .str.replace("ó", "o", regex=False)
    .str.replace("/", " ", regex=False)
    .str.replace(";", " ", regex=False)
)

# Identificação dinâmica das colunas relevantes
colunas = list(df_raw.columns)
coluna_cp      = [c for c in colunas if "4 digitos" in c][0]
coluna_data    = [c for c in colunas if "data" in c][0]
coluna_energia = [c for c in colunas if "energia" in c or "ativa" in c][0]

print(f"Registos carregados: {len(df_raw):,}")
print(f"Colunas identificadas — CP: '{coluna_cp}' | Data: '{coluna_data}' | Energia: '{coluna_energia}'")


Registos carregados: 479,379
Colunas identificadas — CP: 'codigo postal 4 digitos' | Data: 'data hora' | Energia: 'energia ativa (kwh)'


## 2. Filtragem Geográfica — Aveiro (3800 e 3810)

In [7]:
# Garantir que o código postal é string e filtrar Aveiro
df_raw[coluna_cp] = df_raw[coluna_cp].astype(str).str.strip()
df_aveiro = df_raw[
    df_raw[coluna_cp].str.startswith(("3800", "3810"))
].copy()

print(f"Registos totais: {len(df_raw):,} | Registos Aveiro: {len(df_aveiro):,}")


Registos totais: 479,379 | Registos Aveiro: 479,379


## 3. Limpeza e Validação

In [14]:
# --- 3.1 Converter tipos ---
df_aveiro["data_parsed"]    = pd.to_datetime(df_aveiro[coluna_data], errors="coerce")
df_aveiro["energia_parsed"] = pd.to_numeric(df_aveiro[coluna_energia], errors="coerce")

# --- 3.2 Registar e remover nulos ---
nulos_data    = df_aveiro["data_parsed"].isna().sum()
nulos_energia = df_aveiro["energia_parsed"].isna().sum()
print(f"Nulos em data: {nulos_data} | Nulos em energia: {nulos_energia}")
df_aveiro = df_aveiro.dropna(subset=["data_parsed", "energia_parsed"])

# --- 3.3 Remover duplicados exatos ---
duplicados = df_aveiro.duplicated(subset=["data_parsed", coluna_cp]).sum()
print(f"Duplicados removidos: {duplicados}")
df_aveiro = df_aveiro.drop_duplicates(subset=["data_parsed", coluna_cp])

# --- 3.4 Rejeitar valores fisicamente impossíveis (energia negativa) ---
negativos = (df_aveiro["energia_parsed"] < 0).sum()
print(f"Valores negativos removidos: {negativos}")
df_aveiro = df_aveiro[df_aveiro["energia_parsed"] >= 0]

# --- 3.5 Ordenar por data ---
df_aveiro = df_aveiro.sort_values("data_parsed").reset_index(drop=True)

print(f"Registos após limpeza: {len(df_aveiro):,}")
print(f"Janela temporal: {df_aveiro['data_parsed'].min()} → {df_aveiro['data_parsed'].max()}")


Nulos em data: 0 | Nulos em energia: 0
Duplicados removidos: 0
Valores negativos removidos: 0
Registos após limpeza: 1,392
Janela temporal: 2024-02-01 00:00:00+00:00 → 2024-02-29 23:00:00+00:00


## 4. Deteção e Tratamento de Outliers
Utilizamos o método **IQR (Interquartile Range)**: valores abaixo de Q1 − 1.5×IQR
ou acima de Q3 + 1.5×IQR são considerados outliers e substituídos pela mediana da
mesma hora do dia — preservando o perfil intra-diário.

In [15]:
# Calcular limites IQR
Q1  = df_aveiro["energia_parsed"].quantile(0.25)
Q3  = df_aveiro["energia_parsed"].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers_mask = (
    (df_aveiro["energia_parsed"] < lim_inf) |
    (df_aveiro["energia_parsed"] > lim_sup)
)
print(f"Outliers detetados: {outliers_mask.sum()} ({outliers_mask.mean()*100:.2f}% dos registos)")

# Substituir outliers pela mediana da mesma hora do dia
df_aveiro["hora"] = df_aveiro["data_parsed"].dt.hour
mediana_por_hora  = df_aveiro.groupby("hora")["energia_parsed"].transform("median")
df_aveiro.loc[outliers_mask, "energia_parsed"] = mediana_por_hora[outliers_mask]

print("Outliers substituídos pela mediana horária correspondente.")


Outliers detetados: 142 (10.20% dos registos)
Outliers substituídos pela mediana horária correspondente.


## 5. Normalização
Criamos duas versões normalizadas do consumo:
- **Min-Max (0–1):** útil para visualizações e comparações relativas.
- **Z-score (μ=0, σ=1):** útil para algoritmos de ML sensíveis à escala.

In [16]:
e = df_aveiro["energia_parsed"]

# Min-Max
df_aveiro["energia_minmax"] = (e - e.min()) / (e.max() - e.min())

# Z-score
df_aveiro["energia_zscore"] = (e - e.mean()) / e.std()

print("Normalização concluída.")
print(f"Min-Max  → min={df_aveiro['energia_minmax'].min():.3f} | max={df_aveiro['energia_minmax'].max():.3f}")
print(f"Z-score  → μ={df_aveiro['energia_zscore'].mean():.3f}  | σ={df_aveiro['energia_zscore'].std():.3f}")


Normalização concluída.
Min-Max  → min=0.000 | max=1.000
Z-score  → μ=0.000  | σ=1.000


## 6. Engenharia de Features
Criamos variáveis temporais e categóricas que enriquecem o dataset para análise e modelação.

In [11]:
df = df_aveiro.copy()

# --- Temporais ---
df["ano"]            = df["data_parsed"].dt.year
df["mes"]            = df["data_parsed"].dt.month
df["dia"]            = df["data_parsed"].dt.day
df["hora"]           = df["data_parsed"].dt.hour
df["dia_semana"]     = df["data_parsed"].dt.dayofweek   # 0=Segunda, 6=Domingo
df["dia_semana_nome"]= df["data_parsed"].dt.day_name(locale="pt_PT.UTF-8").str[:3]
df["semana_ano"]     = df["data_parsed"].dt.isocalendar().week.astype(int)
df["trimestre"]      = df["data_parsed"].dt.quarter

# --- Flags binárias ---
df["fim_de_semana"]  = (df["dia_semana"] >= 5).astype(int)  # 1 = Sáb ou Dom

# --- Estação do ano (meteorológica) ---
def estacao(mes):
    if mes in [12, 1, 2]:  return "Inverno"
    elif mes in [3, 4, 5]: return "Primavera"
    elif mes in [6, 7, 8]: return "Verão"
    else:                  return "Outono"

df["estacao"] = df["mes"].apply(estacao)

# --- Período do dia ---
def periodo_dia(hora):
    if   0  <= hora < 6:  return "Madrugada"
    elif 6  <= hora < 12: return "Manhã"
    elif 12 <= hora < 18: return "Tarde"
    else:                 return "Noite"

df["periodo_dia"] = df["hora"].apply(periodo_dia)

# --- Hora de ponta vs. vazio (tarifas PT) ---
# Ponta: 09h–12h e 18h–21h dias úteis | Vazio: restante
def tarifa(row):
    if row["fim_de_semana"] == 1:
        return "Vazio"
    h = row["hora"]
    if (9 <= h < 12) or (18 <= h < 21):
        return "Ponta"
    elif (7 <= h < 9) or (12 <= h < 18) or (21 <= h < 22):
        return "Cheio"
    else:
        return "Vazio"

df["periodo_tarifario"] = df.apply(tarifa, axis=1)

print("Features criadas:", [c for c in df.columns if c not in df_aveiro.columns])


Features criadas: ['ano', 'mes', 'dia', 'dia_semana', 'dia_semana_nome', 'semana_ano', 'trimestre', 'fim_de_semana', 'estacao', 'periodo_dia', 'periodo_tarifario']


## 7. Exportação do Dataset Final

In [13]:
# Selecionar e ordenar as colunas finais para exportação
colunas_finais = [
    "data_parsed",
    "ano", "mes", "dia", "hora",
    "dia_semana", "dia_semana_nome", "semana_ano", "trimestre",
    "fim_de_semana", "estacao", "periodo_dia", "periodo_tarifario",
    "energia_parsed",   # valor bruto limpo (kWh)
    "energia_minmax",   # normalizado 0–1
    "energia_zscore",   # normalizado z-score
]

df_final = df[colunas_finais].copy()
df_final = df_final.rename(columns={"data_parsed": "data_hora", "energia_parsed": "energia_kwh"})

df_final.to_csv("consumo_aveiro_preprocessado.csv", index=False)

print("--- Dataset Final Exportado ---")
print(f"Ficheiro: consumo_aveiro_preprocessado.csv")
print(f"Registos: {len(df_final):,} | Colunas: {len(df_final.columns)}")
print(f"Janela: {df_final['data_hora'].min()} → {df_final['data_hora'].max()}")
print("Amostra das primeiras 3 linhas:")
df_final.head(3)


--- Dataset Final Exportado ---
Ficheiro: consumo_aveiro_preprocessado.csv
Registos: 1,392 | Colunas: 16
Janela: 2024-02-01 00:00:00+00:00 → 2024-02-29 23:00:00+00:00
Amostra das primeiras 3 linhas:


,data_hora,ano,mes,dia,hora,dia_semana,dia_semana_nome,semana_ano,trimestre,fim_de_semana,estacao,periodo_dia,periodo_tarifario,energia_kwh,energia_minmax,energia_zscore
0,2024-02-01 00:00:00+00:00,2024,2,1,0,3,Qui,5,1,0,Inverno,Madrugada,Vazio,32.718826,0.250976,-0.186685
1,2024-02-01 00:00:00+00:00,2024,2,1,0,3,Qui,5,1,0,Inverno,Madrugada,Vazio,65.113818,0.526092,1.156090
2,2024-02-01 01:00:00+00:00,2024,2,1,1,3,Qui,5,1,0,Inverno,Madrugada,Vazio,18.077516,0.126635,-0.793568
